In [1]:
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
client = OpenAI()  # 无报错即可

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
files = reader.read()
documents = [f.parse() for f in files]
len(documents)  # → Q1 答案

72

In [3]:
from minsearch import Index
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
index.fit(documents)
results = index.search(
    "How does the agentic loop keep calling the model until it stops?"
)
results[0]["filename"]  # → Q2 答案

'01-agentic-rag/lessons/14-agentic-loop.md'

In [5]:
from rag_helper import RAGBase
query = "How does the agentic loop keep calling the model until it stops?"
assistant = RAGBase(
    index=index,           # Q2 里已经 fit 过的 index
    llm_client=client,     # cell 0 里的 OpenAI client
)
answer, usage = assistant.rag(query)
print(answer)
print(usage)
usage.input_tokens   # → Q3 答案

It keeps calling the model in a `while True` loop, and after each response it checks whether the model returned any `function_call` items.

- If there are function calls, the code runs the tool, appends the tool output to `messages`, and loops again.
- If there are no function calls, it `break`s out of the loop.

So the stop condition is: **no function calls in the model’s response**.
ResponseUsage(input_tokens=7111, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=94, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7205)


7111

In [7]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)  # → Q4 答案

295

In [10]:
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunk_index.fit(chunks)

In [11]:
assistant = RAGBase(
    index=chunk_index,          
    llm_client=client,     
)
answer, usage = assistant.rag(query)
print(answer)
print(usage)
usage.input_tokens

It keeps calling the model inside a `while True` loop. After each model response, the code checks whether any `function_call` items were returned.

- If there is a function call, it runs the tool, appends the tool result to `messages`, and loops again.
- If there are no function calls in that turn, it breaks out of the loop.

So the stop condition is: **no function calls in the latest response**.
ResponseUsage(input_tokens=2294, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=95, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2389)


2294

In [13]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat.interface import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

search_count = 0

def search(query: str) -> list:
    """Search course lesson chunks for content matching the query."""
    global search_count
    search_count += 1
    return chunk_index.search(query, num_results=5)

agent_tools = Tools()
agent_tools.add_tool(search)

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=(
        "You're a course teaching assistant. Answer the student's question "
        "using the search tool. Make multiple searches with different "
        "keywords before answering."
    ),
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini"),
)

result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

print("search called:", search_count)  # → Q6 答案

-> Response received


-> Response received


search called: 3
